In [1]:
"""
@author: Zilan Cheng
@note: This code is modified based on Zongyi Li's original implementation of Fourier Neural Operators.
"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
torch.cuda.set_device(1)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels,out_channels, modes,basis, size):
        super(SpectralConv1d, self).__init__()

        """
        1D integration layer. It does POD transform, linear transform, and Inverse POD transform.    
        """
        self.in_channels=in_channels
        self.out_channels=out_channels     
        self.modes = modes
        self.size = size
        self.basis = basis.to(torch.float32)[:,:self.modes]

        self.scale = (1 / (in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes, dtype=torch.float))


    def forward(self, x):
        batchsize = x.shape[0]
        x_1  = torch.einsum("bwx,xf->bwf",x,self.basis) # f refers to coefficients space,  this step projects from physical space to POD basis
        x_1 = torch.einsum("bif,iof->bof",x_1,self.weights1) # this step defines a diagonal operator 
        x_out  = torch.einsum("bwf,xf->bwx",x_1,self.basis) # f refers to coefficients space, this step recovers from POD basis to physical space
        return x_out

In [3]:
def get_grid(shape, device):
        batchsize, size_x = shape[0], shape[1]
        gridx = torch.tensor(np.linspace(0, 1, size_x))
        gridx = gridx.reshape(1, size_x, 1).repeat([batchsize, 1, 1])
        return gridx.to(device)

In [4]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv1d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv1d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [5]:
class PODNO1d(nn.Module):
    def __init__(self, modes, width,size,basis):
        super(PODNO1d, self).__init__()

        self.modes = modes
        self.width = width
        self.size = size
        self.basis = torch.tensor(basis)

        self.p = nn.Linear(2, self.width) # input channel_dim is 2: (u0(x), x)
        
        self.conv0 = SpectralConv1d(self.width, self.width,self.modes, self.basis,  self.size)
        self.conv1 = SpectralConv1d(self.width, self.width,self.modes, self.basis, self.size)
        self.conv2 = SpectralConv1d(self.width, self.width,self.modes, self.basis, self.size)
        self.conv3 = SpectralConv1d(self.width, self.width,self.modes, self.basis,  self.size)
        self.mlp0 = MLP(self.width, self.width, self.width)
        self.mlp1 = MLP(self.width, self.width, self.width)
        self.mlp2 = MLP(self.width, self.width, self.width)
        self.mlp3 = MLP(self.width, self.width, self.width)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width*2)  # output channel_dim is 1: u1(x)

        

    def forward(self, x):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x=x.to(torch.float32)
        x = self.p(x)
        x = x.permute(0, 2, 1)

        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x = x1 + x2

        x = self.q(x)
        x = x.permute(0, 2, 1)
        return x

In [6]:
################################################################
#  configurations
################################################################
ntrain = 900
ntest = 100
nsnap = 900

modes = 16
width = 32

size=1024

batch_size = 20
learning_rate = 0.001
epochs = 1000
iterations = epochs*(ntrain//batch_size)

In [7]:
################################################################
# dataloader
################################################################
data = np.load("../data/sum_sin/sine_data.npz", allow_pickle=True)

X = data["input"]    
Y = data["output"]
x_train = X[:ntrain,  :] 
y_train = Y[:ntrain, :] 

# x_snap = X[:nsnap,  :]
# y_snap = Y[:nsnap,:]

x_test  = X[-ntest:, :]
y_test  = Y[-ntest:, :]

x_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)

x_test  = torch.tensor(x_test,  dtype=torch.float32).unsqueeze(-1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(-1)

x_train = x_train.reshape(ntrain,size,1)
x_test = x_test.reshape(ntest,size,1)
y_train = y_train.reshape(ntrain,size,1)
y_test = y_test.reshape(ntest,size,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)


In [8]:
################################################################
# SVD
################################################################
x_snap = x_train[:nsnap,:,:]
y_snap =y_train[:nsnap,:,:]

basis_0=torch.cat((x_train,y_train),0)
basis_1=basis_0.permute(2,1,0)
basis_1=basis_1.reshape(-1,2*ntrain)
U,S,V=torch.svd(torch.mm(basis_1,basis_1.T))
basis=U.to(device)

In [9]:
################################################################
# training and evaluation
################################################################
model = PODNO1d(modes, width, size, basis)
model=model.to(device)
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

myloss = LpLoss(size_average=False)
y_normalizer.to(device)
for ep in range(epochs):
    model.train()
    train_mse = 0
    train_l2 = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
    
        optimizer.zero_grad()
        out = model(x)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)
        l2 = myloss(out.view(batch_size, -1), y.view(batch_size, -1))
        l2.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += l2.item()
    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            out = y_normalizer.decode(out)
            test_l2 += myloss(out.view(batch_size, -1), y.view(batch_size, -1)).item()

    train_l2/= ntrain
    test_l2 /= ntest

    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Train L2: {train_l2:.6f} | Test L2: {test_l2:.6f}")

/tmp/ipykernel_3169650/2071170047.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.basis = torch.tensor(basis)


80481
Epoch    0 | Train L2: 0.884903 | Test L2: 0.587310
Epoch   50 | Train L2: 0.030743 | Test L2: 0.025773
Epoch  100 | Train L2: 0.019509 | Test L2: 0.017600
Epoch  150 | Train L2: 0.015438 | Test L2: 0.019808
Epoch  200 | Train L2: 0.015943 | Test L2: 0.014016
Epoch  250 | Train L2: 0.013826 | Test L2: 0.011833
Epoch  300 | Train L2: 0.012023 | Test L2: 0.013434
Epoch  350 | Train L2: 0.010764 | Test L2: 0.010497
Epoch  400 | Train L2: 0.008885 | Test L2: 0.011664
Epoch  450 | Train L2: 0.009584 | Test L2: 0.009320
Epoch  500 | Train L2: 0.009442 | Test L2: 0.009160
Epoch  550 | Train L2: 0.006023 | Test L2: 0.007449
Epoch  600 | Train L2: 0.005715 | Test L2: 0.006263
Epoch  650 | Train L2: 0.005349 | Test L2: 0.005510
Epoch  700 | Train L2: 0.003816 | Test L2: 0.004492
Epoch  750 | Train L2: 0.003479 | Test L2: 0.004122
Epoch  800 | Train L2: 0.002954 | Test L2: 0.004020
Epoch  850 | Train L2: 0.002632 | Test L2: 0.003833
Epoch  900 | Train L2: 0.002294 | Test L2: 0.003593
Epoch 